In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/aieducation/what-course-are-you-going-to-take/src/small_bench/checkpoints/post_cell_8.pickle

In [ ]:
%%cudf.pandas.profile
### cell 9 ###

def extract_filter_features(df, col_name, threshold):
    """Extracts features for predicting execution time of filtering operation."""
    # Cache the column to avoid multiple DataFrame.__getitem__ calls
    col = df[col_name]

    # Number of rows
    num_rows = len(df)

    # Column dtype as string
    dtype_str = str(col.dtype)

    # Count nulls and compute null ratio
    null_count = col.isna().sum().item()
    nan_ratio = null_count / num_rows

    # Number of non-null entries
    non_null_count = num_rows - null_count

    # Compute selectivity among non-null values
    if non_null_count > 0:
        # Boolean comparison (< threshold) treats NaNs as False, so no need to dropna
        passing_count = col.lt(threshold).sum().item()
        target_selectivity = passing_count / non_null_count
    else:
        target_selectivity = 0.0

    return {
        "num_rows": num_rows,
        "dtype_str": dtype_str,
        "nan_ratio": nan_ratio,
        "target_selectivity": target_selectivity,
    }

# Example call
extract_filter_features(course, "num_reviews", 10)